# 04 Homework 04 ETM — Node Classification

**Task 5 of the Final Assignment**

This notebook builds a **CellComplex** from the Brownstone floor plan room volumes, assigns
room-type labels and door-type apertures, exports the graph to CSV in the MSD model schema,
then runs the pretrained `msd_node_classifier.pt` to predict each room's type.

## Pipeline
1. Load room OBJs → build CellComplex with room-type labels  
2. Load door OBJs → add as apertures  
3. `Graph.ByTopology(cc, directApertures=True)` → adjacency graph through doors  
4. Compute zoning and connectivity one-hot features per node/edge  
5. Export to CSV (MSD schema)  
6. Load with `PyG.ByCSVPath`, load pretrained model, predict  
7. Visualise true vs predicted labels

## MSD label → room-type mapping
| `room_type` | `label` | Zoning class |
|---|---:|---|
| `bedroom` | `0` | Private / static |
| `livingroom` | `1` | Living / dynamic |
| `kitchen` | `2` | Living / dynamic |
| `dining` | `3` | Living / dynamic |
| `corridor` | `4` | Living / dynamic |
| `stairs` | `5` | Service / functional |
| `storeroom` | `6` | Service / functional |
| `bathroom` | `7` | Service / functional |
| `balcony` | `8` | Outdoor / semi-outdoor |

**Room OBJs:** `Homework04/Objects/*.obj`  
**Apertures:** `door2.obj` (door) · `Passage Door.obj` (passage)  
**Reference:** instructor @channel pattern + HW02 `Cell.ByFaces` approach

## 1. Imports

In [1]:
from topologicpy.Vertex import Vertex
from topologicpy.Edge import Edge
from topologicpy.Wire import Wire
from topologicpy.Face import Face
from topologicpy.Cell import Cell
from topologicpy.Cluster import Cluster
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary
from topologicpy.Graph import Graph
from topologicpy.Helper import Helper
from topologicpy.Color import Color
from topologicpy.PyG import PyG
import pandas as pd
import numpy as np
import os
from collections import Counter

c:\Users\etmaglari\IAAC\etmaglari_gML\.gmlenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Version check

In [2]:
print("This notebook requires topologicpy 0.9.43 or newer.")
print(Helper.Version())

This notebook requires topologicpy 0.9.43 or newer.
The version that you are using (0.9.43) is OLDER than the latest version (0.9.50) from PyPI. Please consider upgrading to the latest version.


## 3. Renderer

In [3]:
renderer = "vscode"

## 4. Room-type and door-type mappings

### Zoning classes (for node features)
| Zoning | One-hot index | Room types |
|---|---:|---|
| Private/static | `0` | bedroom |
| Living/dynamic | `1` | livingroom, kitchen, dining, corridor |
| Service/functional | `2` | stairs, storeroom, bathroom |
| Outdoor/semi-outdoor | `3` | balcony |

### Connectivity classes (for node and edge features)
| Connection type | One-hot index | Meaning |
|---|---:|---|
| passage | `0` | Open opening / corridor connection |
| door | `1` | Standard interior door |
| entrance_door | `2` | Exterior / entrance door |

## 4. MSD label and feature mappings

In [4]:
ROOM_LABEL = {
    "bedroom": 0, "livingroom": 1, "kitchen": 2, "dining": 3,
    "corridor": 4, "stairs": 5, "storeroom": 6, "bathroom": 7, "balcony": 8,
}
ZONING = {
    "bedroom":   [1,0,0,0], "livingroom": [0,1,0,0], "kitchen":   [0,1,0,0],
    "dining":    [0,1,0,0], "corridor":   [0,1,0,0], "stairs":    [0,0,1,0],
    "storeroom": [0,0,1,0], "bathroom":   [0,0,1,0], "balcony":   [0,0,0,1],
}
NODE_CONNECTIVITY = {
    "bedroom":   [0,1,0], "livingroom": [0,1,0], "kitchen":  [0,1,0],
    "dining":    [0,1,0], "corridor":   [1,0,0], "stairs":   [1,0,0],
    "storeroom": [0,1,0], "bathroom":   [0,1,0], "balcony":  [0,0,0],
}
DOOR_CONNECTIVITY = {
    "passage":       [1,0,0],
    "door":          [0,1,0],
    "entrance_door": [0,0,1],
}
ROOM_COLOR = {
    "bedroom":   "#4E79A7", "livingroom": "#F28E2B", "kitchen":   "#E15759",
    "dining":    "#76B7B2", "corridor":   "#59A14F", "stairs":    "#EDC948",
    "storeroom": "#B07AA1", "bathroom":   "#FF9DA7", "balcony":   "#9C755F",
    "unknown":   "#AAAAAA",
}
print("Mappings loaded.")

Mappings loaded.


## 5. Paths

In [5]:
\
OBJECTS_DIR  = r"C:\Users\etmaglari\IAAC\etmaglari_gML\Homework04\Objects"
MODEL_PATH   = r"C:\Users\etmaglari\IAAC\etmaglari_gML\S0 Classes\msd-main\msd_node_classifier.pt"
DATASET_PATH = r"C:\Users\etmaglari\IAAC\etmaglari_gML\Homework04\dataset_04B"
os.makedirs(DATASET_PATH, exist_ok=True)
print("Objects :", OBJECTS_DIR)
print("Model   :", MODEL_PATH)
print("Dataset :", DATASET_PATH)

Objects : C:\Users\etmaglari\IAAC\etmaglari_gML\Homework04\Objects
Model   : C:\Users\etmaglari\IAAC\etmaglari_gML\S0 Classes\msd-main\msd_node_classifier.pt
Dataset : C:\Users\etmaglari\IAAC\etmaglari_gML\Homework04\dataset_04B


## 6. Load room OBJs and build cells

Each OBJ file contains one room type (multiple objects = multiple floor levels).  
`Topology.ByOBJPath(transposeAxes=True)` converts Rhino Y-up → Z-up.  
`Cell.ByFaces` builds a solid cell at progressively loose tolerances (HW02 approach).

Each OBJ file represents one room type. We import the geometry, extract the enclosed cell
volumes, and create **selector vertices** (internal points) carrying `room_type`, `label`,
`cell_color`, `zoning`, and `connectivity` dictionaries. These selectors are later used to
transfer labels onto the merged CellComplex cells.

In [6]:
ROOM_FILES = {
    "Bedroom":     ("bedroom",    os.path.join(OBJECTS_DIR, "Bedroom.obj")),
    "Living room": ("livingroom", os.path.join(OBJECTS_DIR, "Living room.obj")),
    "Kitchen":     ("kitchen",    os.path.join(OBJECTS_DIR, "Kitchen.obj")),
    "Corridor":    ("corridor",   os.path.join(OBJECTS_DIR, "Corridor.obj")),
    "Stair":       ("stairs",     os.path.join(OBJECTS_DIR, "Stair.obj")),
    "Bathroom":    ("bathroom",   os.path.join(OBJECTS_DIR, "Bathroom.obj")),
}

def build_cell(faces):
    for tol in [0.001, 0.005, 0.01, 0.05, 0.1]:
        c = Cell.ByFaces(faces, tolerance=tol)
        if c is not None:
            return c
    return None

all_cells = []
selectors = []

for display_name, (room_type, obj_path) in ROOM_FILES.items():
    if not os.path.exists(obj_path):
        print(f"  [SKIP] not found: {obj_path}")
        continue

    objs = Topology.ByOBJPath(obj_path, transposeAxes=True)
    if not isinstance(objs, list):
        objs = [objs] if objs else []

    label  = ROOM_LABEL[room_type]
    zoning = ZONING[room_type]
    conn   = NODE_CONNECTIVITY[room_type]
    color  = ROOM_COLOR[room_type]

    n_ok = 0
    for obj in objs:
        if obj is None:
            continue
        faces = Topology.Faces(obj) or []
        if len(faces) < 4:
            continue
        c = build_cell(faces)
        if c is None:
            continue
        iv = Topology.InternalVertex(c)
        d  = Dictionary.ByKeysValues(
            ["room_type", "label", "cell_color",
             "feat_zoning_type_0", "feat_zoning_type_1",
             "feat_zoning_type_2", "feat_zoning_type_3",
             "feat_connectivity_0", "feat_connectivity_1",
             "feat_connectivity_2"],
            [room_type, label, color,
             zoning[0], zoning[1], zoning[2], zoning[3],
             conn[0], conn[1], conn[2]]
        )
        iv = Topology.SetDictionary(iv, d)
        selectors.append(iv)
        all_cells.append(c)
        n_ok += 1

    status = f"cells={n_ok}" if n_ok else "[SKIP] no cells built"
    print(f"  {display_name:20s} -> {room_type:12s}  label={label}  {status}")

print(f"\nTotal: {len(all_cells)} cells, {len(selectors)} selectors")

  Bedroom              -> bedroom       label=0  cells=5
  Living room          -> livingroom    label=1  cells=2
  Kitchen              -> kitchen       label=2  cells=1
  Corridor             -> corridor      label=4  cells=5
  Stair                -> stairs        label=5  cells=4
  Bathroom             -> bathroom      label=7  cells=6

Total: 23 cells, 23 selectors


## 7. Build CellComplex and transfer room-type dictionaries

All room cells are merged into a single `CellComplex`. Then
`Topology.TransferDictionariesBySelectors` assigns the room-type dictionaries from the
selector vertices to the CellComplex cells.

In [7]:
cc = Topology.SelfMerge(Cluster.ByTopologies(all_cells))
print("Topology type :", Topology.TypeAsString(cc))
print("Cells :", len(Topology.Cells(cc) or []))
print("Faces :", len(Topology.Faces(cc) or []))

cc = Topology.TransferDictionariesBySelectors(cc, selectors, tranCells=True, tolerance=0.1)

counts, unlabelled = Counter(), 0
for cell in (Topology.Cells(cc) or []):
    d  = Topology.Dictionary(cell)
    rt = Dictionary.ValueAtKey(d, "room_type")
    if rt:
        counts[rt] += 1
    else:
        unlabelled += 1

print("\nRoom distribution:")
for rt, n in sorted(counts.items(), key=lambda x: ROOM_LABEL.get(x[0], 99)):
    print(f"  {rt:15s}  label={ROOM_LABEL[rt]}  count={n}")
if unlabelled:
    print(f"  *** {unlabelled} unlabelled cell(s)")
else:
    print("  All cells labelled.")

Topology type : CellComplex
Cells : 36
Faces : 228

Room distribution:
  bedroom          label=0  count=5
  livingroom       label=1  count=2
  kitchen          label=2  count=1
  corridor         label=4  count=5
  stairs           label=5  count=4
  bathroom         label=7  count=6
  *** 13 unlabelled cell(s)


## 8. Visualise CellComplex coloured by room type

In [8]:
cc_cells = Topology.Cells(cc) or []
display_faces = []
for cell in cc_cells:
    d     = Topology.Dictionary(cell)
    color = Dictionary.ValueAtKey(d, "cell_color") or "#AAAAAA"
    for f in (Topology.Faces(cell) or []):
        f = Topology.SetDictionary(
            f, Dictionary.ByKeysValues(["cell_color"], [color]))
        display_faces.append(f)

Topology.Show(
    display_faces,
    faceColorKey="cell_color",
    faceOpacity=0.6,
    backgroundColor="white",
    width=900, height=700,
    renderer=renderer
)

## 9. Load door OBJs as apertures

Doors are modelled as planar face objects. We collect the faces from `door.obj` and
`Entrance door.obj`, tag each face with a `door_type` dictionary, then pass them as
apertures to `Topology.AddApertures`.

The apertures are placed on whichever CellComplex face they coincide with (within tolerance).
When `Graph.ByTopology(cc, directApertures=True)` is later called, it creates graph edges
only between cells whose shared face carries an aperture — modelling room connectivity
through actual door openings.

In [9]:
DOOR_FILES = {
    "door":    os.path.join(OBJECTS_DIR, "door2.obj"),
    "passage": os.path.join(OBJECTS_DIR, "Passage Door.obj"),
}

apertures = []

for door_type, obj_path in DOOR_FILES.items():
    if not os.path.exists(obj_path):
        print(f"  [SKIP] not found: {obj_path}")
        continue

    conn = DOOR_CONNECTIVITY[door_type]
    objs = Topology.ByOBJPath(obj_path, transposeAxes=True)
    if not isinstance(objs, list):
        objs = [objs] if objs else []

    faces_for_type = []
    for obj in objs:
        if obj is None:
            continue
        faces = Topology.Faces(obj) or []
        if faces:
            faces_for_type.extend(faces)
        else:
            for w in (Topology.Wires(obj) or []):
                f = Face.ByWire(w)
                if f is not None:
                    faces_for_type.append(f)

    for f in faces_for_type:
        d = Dictionary.ByKeysValues(
            ["door_type",
             "feat_connectivity_0", "feat_connectivity_1", "feat_connectivity_2"],
            [door_type, conn[0], conn[1], conn[2]]
        )
        f = Topology.SetDictionary(f, d)
        apertures.append(f)

    print(f"  {door_type:20s} -> {len(faces_for_type)} aperture faces")

print(f"\nTotal apertures: {len(apertures)}")

  door                 -> 11 aperture faces
  passage              -> 9 aperture faces

Total apertures: 20


## 10. Add apertures to CellComplex

In [10]:
if not apertures:
    print("No apertures found — graph will use direct cell adjacency.")
else:
    # Try increasingly loose tolerances until all (or max possible) apertures attach.
    # Entrance-door faces on exterior walls will never match an interior CellComplex
    # face — that is expected behaviour (they border outside, not a second room cell).
    best_cc      = cc
    best_matched = 0
    for tol in [0.001, 0.01, 0.05, 0.1]:
        cc_try  = Topology.AddApertures(
            cc, apertures, exclusive=False, subTopologyType="Face", tolerance=tol)
        matched = sum(
            1 for f in (Topology.Faces(cc_try) or [])
            if Topology.Apertures(f)
        )
        print(f"  tolerance={tol:5.3f}  ->  {matched}/{len(apertures)} faces matched")
        if matched > best_matched:
            best_matched = matched
            best_cc      = cc_try
        if matched == len(apertures):
            break

    cc            = best_cc
    faces_with_ap = best_matched
    unmatched     = len(apertures) - faces_with_ap
    print(f"\nFaces carrying apertures : {faces_with_ap}")
    if unmatched:
        print(f"Unmatched apertures      : {unmatched}  "
              "(entrance / exterior-wall doors — OK, they don't connect two rooms)")

  tolerance=0.001  ->  19/20 faces matched
  tolerance=0.010  ->  19/20 faces matched
  tolerance=0.050  ->  19/20 faces matched
  tolerance=0.100  ->  19/20 faces matched

Faces carrying apertures : 19
Unmatched apertures      : 1  (entrance / exterior-wall doors — OK, they don't connect two rooms)


## 11. Build the room adjacency graph

`Graph.ByTopology` with `directApertures=True` creates:
- One **vertex** per cell (room)  
- One **edge** between two cells whose shared face carries a door aperture

This exactly mirrors the MSD dataset construction.

In [11]:
graph = Graph.ByTopology(cc, direct=False, directApertures=True)

n_v = len(Graph.Vertices(graph) or [])
n_e = len(Graph.Edges(graph) or [])
print(f"Graph: {n_v} vertices (rooms), {n_e} edges (door connections)")

if n_e == 0:
    print("No aperture edges — falling back to direct cell adjacency.")
    graph = Graph.ByTopology(cc, direct=True, directApertures=False)
    n_v = len(Graph.Vertices(graph) or [])
    n_e = len(Graph.Edges(graph) or [])
    print(f"Fallback graph: {n_v} vertices, {n_e} edges")

Graph: 36 vertices (rooms), 18 edges (door connections)


## 12. Verify graph vertex and edge dictionaries

In [12]:
vertices = Graph.Vertices(graph) or []
edges    = Graph.Edges(graph) or []

print("Sample vertex dictionaries (first 6):")
for v in vertices[:6]:
    d  = Topology.Dictionary(v)
    rt = Dictionary.ValueAtKey(d, "room_type")
    lb = Dictionary.ValueAtKey(d, "label")
    z0 = Dictionary.ValueAtKey(d, "feat_zoning_type_0")
    c0 = Dictionary.ValueAtKey(d, "feat_connectivity_0")
    print(f"  room_type={str(rt):12s}  label={lb}  zoning[0]={z0}  conn[0]={c0}")

print(f"\nSample edge dictionaries (first 5):")
for e in edges[:5]:
    d  = Topology.Dictionary(e)
    dt = Dictionary.ValueAtKey(d, "door_type")
    ks = Dictionary.Keys(d)
    print(f"  door_type={dt}  keys={ks}")

Sample vertex dictionaries (first 6):
  room_type=bedroom       label=0  zoning[0]=1  conn[0]=0
  room_type=corridor      label=4  zoning[0]=0  conn[0]=1
  room_type=bathroom      label=7  zoning[0]=0  conn[0]=0
  room_type=corridor      label=4  zoning[0]=0  conn[0]=1
  room_type=None          label=None  zoning[0]=None  conn[0]=None
  room_type=bedroom       label=0  zoning[0]=1  conn[0]=0

Sample edge dictionaries (first 5):
  door_type=passage  keys=['category', 'door_type', 'dst', 'feat_connectivity_0', 'feat_connectivity_1', 'feat_connectivity_2', 'ontology_class', 'ontology_uri', 'relationship', 'src', 'type']
  door_type=passage  keys=['category', 'door_type', 'dst', 'feat_connectivity_0', 'feat_connectivity_1', 'feat_connectivity_2', 'ontology_class', 'ontology_uri', 'relationship', 'src', 'type']
  door_type=passage  keys=['category', 'door_type', 'dst', 'feat_connectivity_0', 'feat_connectivity_1', 'feat_connectivity_2', 'ontology_class', 'ontology_uri', 'relationship', 'src

## 13. Export CSVs in MSD schema

| File | Columns |
|---|---|
| `graphs.csv` | `graph_id`, `num_nodes` |
| `nodes.csv` | `graph_id`, `node_id`, `label`, features, masks |
| `edges.csv` | `graph_id`, `src_id`, `dst_id`, `feat_connectivity_0..2` |

In [ ]:
def vkey(v, tol=3):
    return tuple(round(x, tol) for x in Vertex.Coordinates(v))

def gv(d, k, default=0):
    val = Dictionary.ValueAtKey(d, k)
    return val if val is not None else default

# --- keep only labelled vertices (drop SelfMerge artifact cells) ---
labelled = []          # (original_index, vertex)
for i, v in enumerate(vertices):
    d  = Topology.Dictionary(v)
    rt = Dictionary.ValueAtKey(d, "room_type")
    if rt is not None:
        labelled.append((i, v))

print(f"Labelled vertices : {len(labelled)} / {len(vertices)}")

# remapped node_id = position in `labelled`
old_to_new = {old_i: new_i for new_i, (old_i, _) in enumerate(labelled)}
coord_to_new = {vkey(v): old_to_new[old_i] for new_i, (old_i, v) in enumerate(labelled)}

# nodes.csv
nodes_rows = []
for new_i, (old_i, v) in enumerate(labelled):
    d = Topology.Dictionary(v)
    nodes_rows.append({
        "graph_id": 0, "node_id": new_i,
        "label":               int(gv(d, "label", 0)),
        "feat_zoning_type_0":  int(gv(d, "feat_zoning_type_0")),
        "feat_zoning_type_1":  int(gv(d, "feat_zoning_type_1")),
        "feat_zoning_type_2":  int(gv(d, "feat_zoning_type_2")),
        "feat_zoning_type_3":  int(gv(d, "feat_zoning_type_3")),
        "feat_connectivity_0": int(gv(d, "feat_connectivity_0")),
        "feat_connectivity_1": int(gv(d, "feat_connectivity_1", 1)),
        "feat_connectivity_2": int(gv(d, "feat_connectivity_2")),
        "train_mask": 0, "val_mask": 0, "test_mask": 1,
    })

# edges.csv — only edges where BOTH endpoints are labelled
edges_rows = []
for e in edges:
    sk  = vkey(Edge.StartVertex(e))
    ek  = vkey(Edge.EndVertex(e))
    src = coord_to_new.get(sk)
    dst = coord_to_new.get(ek)
    if src is None or dst is None:
        continue          # skip edges to/from unlabelled artifact cells
    d = Topology.Dictionary(e)
    edges_rows.append({
        "graph_id": 0, "src_id": src, "dst_id": dst,
        "feat_connectivity_0": int(gv(d, "feat_connectivity_0")),
        "feat_connectivity_1": int(gv(d, "feat_connectivity_1", 1)),
        "feat_connectivity_2": int(gv(d, "feat_connectivity_2")),
    })

pd.DataFrame([{"graph_id": 0, "num_nodes": len(nodes_rows)}]).to_csv(
    os.path.join(DATASET_PATH, "graphs.csv"), index=False)
pd.DataFrame(nodes_rows).to_csv(
    os.path.join(DATASET_PATH, "nodes.csv"), index=False)
pd.DataFrame(edges_rows).to_csv(
    os.path.join(DATASET_PATH, "edges.csv"), index=False)

print(f"graphs.csv : 1 graph")
print(f"nodes.csv  : {len(nodes_rows)} nodes  (labelled rooms only)")
print(f"edges.csv  : {len(edges_rows)} edges")
print(f"Saved to   : {DATASET_PATH}")

## 14. Inspect exported CSVs

In [14]:
nodes_df = pd.read_csv(os.path.join(DATASET_PATH, "nodes.csv"))
edges_df = pd.read_csv(os.path.join(DATASET_PATH, "edges.csv"))
print("nodes.csv")
print(nodes_df.to_string(index=False))
print()
print("edges.csv")
print(edges_df.to_string(index=False))

nodes.csv
 graph_id  node_id  label  feat_zoning_type_0  feat_zoning_type_1  feat_zoning_type_2  feat_zoning_type_3  feat_connectivity_0  feat_connectivity_1  feat_connectivity_2  train_mask  val_mask  test_mask
        0        0      0                   1                   0                   0                   0                    0                    1                    0           0         0          1
        0        1      4                   0                   1                   0                   0                    1                    0                    0           0         0          1
        0        2      7                   0                   0                   1                   0                    0                    1                    0           0         0          1
        0        3      4                   0                   1                   0                   0                    1                    0                    0           0  

## 15. Load dataset into PyG

In [15]:
pyg = PyG.ByCSVPath(
    path=DATASET_PATH,
    level="node",
    task="classification",
    graphLabelType="categorical",
    nodeLabelType="categorical",
    edgeLabelType="categorical",
)
print(pyg)

## 16. Load the pretrained MSD node classifier

In [16]:
pyg.LoadModel(MODEL_PATH)
print("Model loaded.")

Model loaded.


## 17. Predict room types

In [17]:
def to_class(val):
    a = np.squeeze(np.asarray(val))
    if a.ndim == 0: return int(a)
    if a.ndim == 1: return int(np.argmax(a)) if a.size > 1 else int(a[0])
    raise ValueError(f"Unexpected shape {a.shape}")

report    = pyg.Predict(split="all", return_probs=True, attach_to_data=True)
pred_list = report["pred"]
true_list = report["y_true"]
prob_list = report.get("prob", None)

LABEL_NAME = {v: k for k, v in ROOM_LABEL.items()}

rows = []
for g_idx, data in enumerate(pyg.data_list):
    gid   = int(data.graph_id.item()) if hasattr(data, "graph_id") else g_idx
    n     = data.num_nodes
    gp    = np.asarray(pred_list[g_idx])
    gt    = np.asarray(true_list[g_idx])
    gprob = np.asarray(prob_list[g_idx]) if prob_list else None
    for ni in range(n):
        yt  = to_class(gt[ni])
        yp  = to_class(gp[ni])
        row = {"graph_id": gid, "node_id": ni, "y_true": yt, "y_pred": yp}
        if gprob is not None:
            p = np.squeeze(np.asarray(gprob[ni]))
            if p.ndim == 1 and yp < p.size:
                row["y_pred_prob"] = float(p[yp])
        rows.append(row)

pred_df = pd.DataFrame(rows)
pred_df["true_name"] = pred_df["y_true"].map(LABEL_NAME)
pred_df["pred_name"] = pred_df["y_pred"].map(LABEL_NAME)

PRED_CSV = os.path.join(DATASET_PATH, "node_predictions.csv")
pred_df.to_csv(PRED_CSV, index=False)

correct = (pred_df["y_true"] == pred_df["y_pred"]).sum()
total   = len(pred_df)
print(f"Predictions: {correct}/{total} correct = {correct/total:.1%}")
print(f"Saved: {PRED_CSV}")
print(pred_df[["node_id","true_name","pred_name"]].to_string(index=False))

Predictions: 14/36 correct = 38.9%
Saved: C:\Users\etmaglari\IAAC\etmaglari_gML\Homework04\dataset_04B\node_predictions.csv
 node_id  true_name pred_name
       0    bedroom   bedroom
       1   corridor    stairs
       2   bathroom  bathroom
       3   corridor    dining
       4    bedroom    stairs
       5    bedroom   bedroom
       6   bathroom  bathroom
       7     stairs    stairs
       8    bedroom   bedroom
       9    bedroom    stairs
      10   bathroom  bathroom
      11     stairs    stairs
      12   corridor    dining
      13    bedroom    stairs
      14    bedroom   bedroom
      15   bathroom  bathroom
      16    bedroom    stairs
      17    bedroom    stairs
      18   bathroom    stairs
      19   corridor    dining
      20    bedroom    stairs
      21 livingroom   kitchen
      22    bedroom    stairs
      23    bedroom    stairs
      24    bedroom    stairs
      25     stairs    stairs
      26   corridor    dining
      27    bedroom    stairs
      

## 18. Visualise true vs predicted labels

Misclassified nodes shown in **red** (size 30).  
Correctly classified nodes use a colour scale.

In [ ]:
# `labelled` is the filtered list from §13: [(old_idx, vertex), ...]
# new node_id = position in labelled, which matches pred_df["node_id"]
pred_lookup = pred_df.set_index("node_id")[["y_true", "y_pred"]].to_dict("index")

for new_i, (old_i, v) in enumerate(labelled):
    row = pred_lookup.get(new_i, {"y_true": 0, "y_pred": 0})
    yt  = int(row["y_true"])
    yp  = int(row["y_pred"])
    d   = Topology.Dictionary(v)
    if yt != yp:
        sz = 30
        tc = pc = "red"
    else:
        sz = 14
        tc = Color.ByValueInRange(yt, minValue=0, maxValue=8)
        pc = Color.ByValueInRange(yp, minValue=0, maxValue=8)
    d = Dictionary.SetValuesAtKeys(
        d,
        ["true_color", "pred_color", "node_size", "true_label", "pred_label"],
        [tc, pc, sz, yt, yp]
    )
    v = Topology.SetDictionary(v, d)

g_vis = Graph.Reshape(graph)

print("--- True labels ---")
Topology.Show(
    g_vis,
    vertexSize=6, vertexSizeKey="node_size",
    vertexColorKey="true_color",
    showVertexLabel=True, vertexLabelKey="true_label", vertexLabelFontSize=18,
    backgroundColor="white", camera=[0, 0, 3],
    width=900, height=600, renderer=renderer
)

print("--- Predicted labels ---")
Topology.Show(
    g_vis,
    vertexSize=6, vertexSizeKey="node_size",
    vertexColorKey="pred_color",
    showVertexLabel=True, vertexLabelKey="pred_label", vertexLabelFontSize=18,
    backgroundColor="white", camera=[0, 0, 3],
    width=900, height=600, renderer=renderer
)